# ClipCompass Research Analysis 🧭

This notebook analyzes the results from the ClipCompass video processing pipeline. Use this to generate graphs and insights for your research.

## Setup

In [6]:
import os
import sys
import json
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Robustly find 'backend' directory
current_dir = Path(os.getcwd())
if current_dir.name == 'backend':
    project_root = current_dir
elif (current_dir / 'backend').exists():
    project_root = current_dir / 'backend'
else:
    # Fallback: look for 'app' directory
    project_root = current_dir

sys.path.append(str(project_root))
print(f"📂 Project Root: {project_root}")

# Config
DB_PATH = str(project_root / "clipcompass.db")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

ModuleNotFoundError: No module named 'pandas'

## 1. Load Data

Load video metadata, extracted frames, and transcript segments from the SQLite database.

In [ ]:
def load_data():
    conn = sqlite3.connect(DB_PATH)
    
    # Load Videos
    videos = pd.read_sql_query("SELECT * FROM videos", conn)
    
    # Load Frames (with tags)
    frames = pd.read_sql_query("SELECT * FROM frames", conn)
    
    # Load Transcripts
    transcripts = pd.read_sql_query("SELECT * FROM transcript_segments", conn)
    
    conn.close()
    return videos, frames, transcripts

try:
    df_videos, df_frames, df_transcripts = load_data()
    print(f"✅ Loaded {len(df_videos)} videos")
    print(f"✅ Loaded {len(df_frames)} frames")
    print(f"✅ Loaded {len(df_transcripts)} transcript segments")
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("Make sure you have processed at least one video!")

## 2. Visual Content Analysis

Analyze the visual tags generated by the ResNet50 model to understand what content is most prevalent in your video dataset.

In [ ]:
# Flatten tags
all_tags = []

if not df_frames.empty and 'tags' in df_frames.columns:
    for tags_json in df_frames['tags'].dropna():
        try:
            tags = json.loads(tags_json)
            all_tags.extend(tags)
        except:
            pass

    # Count frequencies
    tag_counts = pd.Series(all_tags).value_counts().head(20)

    # Plot
    plt.figure(figsize=(12, 8))
    sns.barplot(x=tag_counts.values, y=tag_counts.index, palette="viridis")
    plt.title("Top 20 Visual Tags Detected (ResNet50)")
    plt.xlabel("Frequency")
    plt.ylabel("Tag")
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No frame tags found. Process a video first.")

## 3. Audio/Transcript Analysis

Analyze the spoken content. We look at the distribution of speech segment durations.

In [ ]:
if not df_transcripts.empty:
    # Calculate duration
    df_transcripts['duration'] = df_transcripts['end_time'] - df_transcripts['start_time']
    
    # Plot duration distribution
    plt.figure(figsize=(10, 6))
    sns.histplot(df_transcripts['duration'], bins=30, kde=True, color="skyblue")
    plt.title("Distribution of Speech Segment Durations")
    plt.xlabel("Duration (seconds)")
    plt.ylabel("Count")
    plt.show()
    
    print(f"Average Segment Duration: {df_transcripts['duration'].mean():.2f} seconds")
else:
    print("⚠️ No transcripts found.")

## 4. Search Simulation (Latency & Accuracy)

Connect to the Qdrant vector database (if running) and simulate a search to visualize scores.

In [ ]:
from qdrant_client import QdrantClient
from app.config import settings
from app.services.embedder import Embedder
import asyncio

async def simulate_search(query_text):
    # Connect
    try:
        client = QdrantClient(host=settings.qdrant_host, port=settings.qdrant_port, timeout=2)
        client.get_collections()
    except:
        print("⚠️ Qdrant is not running. Cannot simulate search.")
        return

    print(f"🔍 Simulating search for: '{query_text}'")
    
    # Embed
    embedder = Embedder()
    # Note: This loads models, might take a moment
    embedding = await embedder.embed_text([query_text])
    
    # Search Transcripts
    results = client.search(
        collection_name=settings.qdrant_collection_transcripts,
        query_vector=embedding[0].tolist(),
        limit=10
    )
    
    # Visualize Scores
    scores = [r.score for r in results]
    texts = [r.payload.get('text', '')[:50] + "..." for r in results]
    
    if scores:
        plt.figure(figsize=(10, 6))
        sns.barplot(x=scores, y=texts, palette="magma")
        plt.title(f"Vector Similarity Scores for query: '{query_text}'")
        plt.xlabel("Cosine Similarity")
        plt.axvline(x=0.7, color='r', linestyle='--', label='Threshold')
        plt.legend()
        plt.show()
    else:
        print("No results found.")

# Run simulation (mock async run in notebook)
# await simulate_search("example query") 
print("Run 'await simulate_search(\"your query\")' to test retrieval")